In [1]:
import pandas as pd
book=pd.read_csv('https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv')

In [2]:
book

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3
...,...,...,...
5976474,49925,510,5
5976475,49925,528,4
5976476,49925,722,4
5976477,49925,949,5


In [3]:
book.shape

(5976479, 3)

In [4]:
book.describe()

,user_id,book_id,rating
count,5.976479e+06,5.976479e+06,5.976479e+06
mean,2.622446e+04,2.006477e+03,3.919866e+00
std,1.541323e+04,2.468499e+03,9.910868e-01
min,1.000000e+00,1.000000e+00,1.000000e+00
25%,1.281300e+04,1.980000e+02,3.000000e+00
50%,2.593800e+04,8.850000e+02,4.000000e+00
75%,3.950900e+04,2.973000e+03,5.000000e+00
max,5.342400e+04,1.000000e+04,5.000000e+00


In [5]:
book.info()

<class 'pandas.DataFrame'>
RangeIndex: 5976479 entries, 0 to 5976478
Data columns (total 3 columns):
 #   Column   Dtype
---  ------   -----
 0   user_id  int64
 1   book_id  int64
 2   rating   int64
dtypes: int64(3)
memory usage: 136.8 MB


In [6]:
book.isnull().sum()

user_id    0
book_id    0
rating     0
dtype: int64

In [7]:
!pip install scikit-surprise


  Obtaining dependency information for scikit-surprise from https://files.pythonhosted.org/packages/01/c6/74c51b462157c054dee4b5a49e669edc8be9eed27d116bd36c4e0ad97447/scikit_surprise-1.1.5-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   - -------------------------------------- 0.0/1.3 MB 495.5 kB/s eta 0:00:03
   ---- ----------------------------------- 0.1/1.3 MB 1.2 MB/s eta 0:00:01
   ----- ---------------------------------- 0.2/1.3 MB 1.3 MB/s eta 0:00:01
   ----- ---------------------------------- 0.2/1.3 MB 1.3 MB/s eta 0:00:01
   ----- ---------------------------------- 0.2/1.3 MB 1.3 MB/s eta 0:00:01
   ----- ---------------------------------- 0.2/1.3 MB 696.3 kB/s eta 0:00:02
   ------ --------------------------------- 0.2/1.3 MB 692.4 kB/s eta 0:00:02
   ------- --------------------

In [10]:
book_sample=book.sample(n=100000,random_state=42)
from surprise import Reader,Dataset
reader=Reader(rating_scale=(1,5))
data=Dataset.load_from_df(book_sample[['user_id','book_id','rating']],reader)

In [15]:
from surprise.model_selection import train_test_split
from surprise import SVD
from surprise import accuracy
trainset,testset=train_test_split(data,test_size=0.2)
algo=SVD(n_factors=100,n_epochs=20,lr_all=0.005,reg_all=0.02)
algo.fit(trainset)
prediction=algo.test(testset)
rmse_score=accuracy.rmse(prediction)

RMSE: 0.9617


In [17]:
from surprise.model_selection import GridSearchCV
param_grid={
    'n_epochs':[10,20,30],
    'lr_all':[0.002,0.005,0.1],
    'reg_all':[0.02,0.01,0.1],
    'n_factors':[50,100,200,400]
}
gs=GridSearchCV(SVD,param_grid,measures=['rmse'],cv=3)
gs.fit(data)
print('best rmse score is: ',gs.best_score['rmse'])
print('best setting for best rmse score is : ', gs.best_params['rmse'])

best rmse score is:  0.953889579772368
best setting for best rmse score is :  {'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.1, 'n_factors': 50}


In [18]:
from surprise import SVD
trainset=data.build_full_trainset()
model=SVD(n_factors=50,n_epochs=30,lr_all=0.005,reg_all=0.1)
model.fit(trainset)
print('SVD model trained succesfully')

SVD model trained succesfully


In [19]:
prediction=model.predict(2,258)
print(prediction)

user: 2          item: 258        r_ui = None   est = 4.18   {'was_impossible': False}
